# Bank Marketing Funnel Analysis — Exploratory Notebook

**Dataset:** UCI Bank Marketing (Moro et al., 2011)  
**Goal:** Build a 5-stage marketing funnel, identify drop-off, segment performance, and recommendations.

Run each cell top-to-bottom. The full scripted pipeline lives in `../src/funnel_analysis.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)

## 1. Load the data

In [ ]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
print(f'Shape: {df.shape}')
print(f'Baseline conversion: {(df.y == "yes").mean():.2%}')
df.head()

In [ ]:
df.describe(include='all').T

## 2. Define the five-stage funnel

| # | Stage | Criterion |
|---|---|---|
| 1 | Targeted  | In the contact list |
| 2 | Reached   | Contact channel known |
| 3 | Engaged   | Call duration > 60s |
| 4 | Qualified | ≤ 3 campaign contacts |
| 5 | Converted | y = yes |

In [ ]:
df['s1_targeted']  = True
df['s2_reached']   = df['contact'].isin(['cellular', 'telephone'])
df['s3_engaged']   = df['s2_reached'] & (df['duration'] > 60)
df['s4_qualified'] = df['s3_engaged']  & (df['campaign'] <= 3)
df['s5_converted'] = (df['y'] == 'yes')

stages = ['s1_targeted', 's2_reached', 's3_engaged', 's4_qualified', 's5_converted']
funnel = pd.DataFrame({
    'stage': ['Targeted', 'Reached', 'Engaged', 'Qualified', 'Converted'],
    'count': [df[s].sum() for s in stages],
})
funnel['pct_of_top'] = (funnel['count'] / funnel['count'].iloc[0] * 100).round(2)
funnel['step_conv']  = (funnel['count'] / funnel['count'].shift(1) * 100).round(2).fillna(100)
funnel['drop_off']   = funnel['count'].shift(1).sub(funnel['count']).fillna(0).astype(int)
funnel

In [ ]:
# Funnel visualization
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3a322a', '#3a322a', '#3a322a', '#b08d3c', '#c1272d']
bars = ax.barh(funnel['stage'][::-1], funnel['count'][::-1], color=colors[::-1])
for bar, pct in zip(bars, funnel['pct_of_top'][::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}  ({pct}%)', va='center', fontsize=10)
ax.set_xlabel('Clients')
ax.set_title('Bank Marketing Funnel — 45,211 clients', fontsize=14, fontweight='bold')
ax.set_xlim(0, funnel['count'].max() * 1.2)
plt.tight_layout()
plt.show()

## 3. Conversion by segment

In [ ]:
def conv_by(col):
    g = df.groupby(col).agg(n=('y', 'size'), conv=('s5_converted', 'sum'))
    g['rate'] = (g['conv'] / g['n'] * 100).round(2)
    return g.sort_values('rate', ascending=False)

for col in ['job', 'month', 'poutcome', 'contact', 'marital', 'education']:
    print(f'\n=== by {col} ===')
    print(conv_by(col))

In [ ]:
# Visualize: conversion by job
job_conv = conv_by('job').reset_index()
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#c1272d' if r > 20 else '#b08d3c' if r > 13 else '#3a322a' for r in job_conv['rate']]
ax.barh(job_conv['job'], job_conv['rate'], color=colors)
ax.axvline(11.7, color='red', linestyle='--', alpha=0.5, label='Baseline (11.7%)')
ax.set_xlabel('Conversion rate (%)')
ax.set_title('Conversion rate by job', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Call duration — the strongest single predictor

In [ ]:
bins = [0, 60, 180, 300, 600, 1200, 10_000]
labels = ['0-60s', '60-180s', '180-300s', '300-600s', '600-1200s', '1200s+']
df['dur_bucket'] = pd.cut(df['duration'], bins=bins, labels=labels, right=False)

dur = df.groupby('dur_bucket', observed=True).agg(
    calls=('y', 'size'),
    conv=('s5_converted', 'sum'),
)
dur['rate'] = (dur['conv'] / dur['calls'] * 100).round(2)
print(dur)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(dur.index.astype(str), dur['rate'], color='#c1272d')
ax.set_ylabel('Conversion rate (%)')
ax.set_title('Conversion by call duration bucket', fontsize=14, fontweight='bold')
for i, v in enumerate(dur['rate']):
    ax.text(i, v + 1, f'{v}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Campaign fatigue

In [ ]:
fat = df.groupby(df['campaign'].clip(upper=10)).agg(
    calls=('y', 'size'),
    conv=('s5_converted', 'sum'),
)
fat['rate'] = (fat['conv'] / fat['calls'] * 100).round(2)
print(fat)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(fat.index, fat['rate'], marker='o', color='#c1272d', linewidth=2.5)
ax.fill_between(fat.index, fat['rate'], alpha=0.1, color='#c1272d')
ax.set_xlabel('Contacts in campaign (clipped at 10)')
ax.set_ylabel('Conversion rate (%)')
ax.set_title('Campaign contact fatigue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Summary

- Baseline conversion is 11.7%, but 28.8% of contacts are lost at Stage 1 to channel attribution failures.
- Previous-success clients convert at 64.7% (5.5× lift) — **the single best targeting lever in the dataset**.
- March, Sep, Oct, Dec convert at 44–52% vs May's ~6–7%. Seasonality is massive.
- Call duration > 10 min = 46% conversion; < 60s = 0.19%. Agent coaching on openers is critical.
- Cap contacts per client at 3 — calls 4+ have diminishing returns.

See `../REPORT.md` for the full write-up and recommendations.